# Project 2: PDF Text Extractor

This notebook demonstrates extracting lines matching specified keywords from a PDF and exporting the results to Excel.

PDFからキーワードを含む行を抽出し、Excelに出力するまでの過程を示します。

---

In [1]:
import sys
import logging
import pandas as pd

sys.path.insert(0, 'src')
from extractor import extract_lines, save_to_excel

print('Setup complete.')

Setup complete.


## 1. Target PDF / 対象PDFの確認

`data/sample.pdf` is a 2-page simulated system log file containing INFO, WARNING, and ERROR lines.

`data/sample.pdf` は2ページのシステムログファイルを模したPDFです。INFO / WARNING / ERROR の3種類のログ行が含まれています。

In [2]:
import pdfplumber

with pdfplumber.open('data/sample.pdf') as pdf:
    total_pages = len(pdf.pages)
    print(f'Pages: {total_pages}')
    print()
    for i, page in enumerate(pdf.pages, 1):
        text = page.extract_text()
        lines = text.strip().split('\n') if text else []
        print(f'--- Page {i} ({len(lines)} lines) ---')
        for line in lines:
            print(f'  {line}')
        print()

Pages: 2

--- Page 1 (11 lines) ---
  System Log Report - 2024-01-15
  2024-01-15 08:00:01 INFO Application started successfully
  2024-01-15 08:00:05 INFO Database connection established
  2024-01-15 08:01:12 WARNING High memory usage detected: 85%
  2024-01-15 08:02:33 ERROR Connection timeout: api.example.com
  2024-01-15 08:03:00 INFO Retry attempt 1 for api.example.com
  2024-01-15 08:03:15 ERROR Connection timeout: api.example.com (retry failed)
  2024-01-15 08:04:00 INFO Switched to fallback endpoint
  2024-01-15 08:05:22 WARNING Disk usage exceeded 90% on /var/log
  2024-01-15 08:06:00 INFO Log rotation triggered
  2024-01-15 08:10:00 INFO Health check passed

--- Page 2 (10 lines) ---
  System Log Report - 2024-01-15 (continued)
  2024-01-15 09:00:00 INFO Scheduled batch job started
  2024-01-15 09:05:44 ERROR File not found: /data/input/sales_2024.csv
  2024-01-15 09:05:45 ERROR Batch job aborted due to missing input file
  2024-01-15 09:06:00 WARNING Email notification faile

---
## 2. Single Keyword Extraction / 単一キーワード抽出

Extract all lines containing `ERROR` (case-insensitive).

`ERROR` を含む行のみを抽出します（大文字小文字を無視）。

In [3]:
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s', force=True)

records_error = extract_lines('data/sample.pdf', ['ERROR'])
df_error = pd.DataFrame(records_error)
print(f'\nFound {len(df_error)} matching lines:')
df_error

INFO: ページ数: 2


INFO: マッチ行数: 4



Found 4 matching lines:


,page,line_num,text
0,1,5,2024-01-15 08:02:33 ERROR Connection timeout: ...
1,1,7,2024-01-15 08:03:15 ERROR Connection timeout: ...
2,2,3,2024-01-15 09:05:44 ERROR File not found: /dat...
3,2,4,2024-01-15 09:05:45 ERROR Batch job aborted du...


---
## 3. Multiple Keywords (OR Search) / 複数キーワード検索

Extract lines containing `ERROR` **or** `WARNING`.

`ERROR` または `WARNING` を含む行をまとめて抽出します（OR条件）。

In [4]:
records_multi = extract_lines('data/sample.pdf', ['ERROR', 'WARNING'])
df_multi = pd.DataFrame(records_multi)
print(f'Found {len(df_multi)} matching lines (ERROR or WARNING):')
df_multi

INFO: ページ数: 2


INFO: マッチ行数: 8


Found 8 matching lines (ERROR or WARNING):


,page,line_num,text
0,1,4,2024-01-15 08:01:12 WARNING High memory usage ...
1,1,5,2024-01-15 08:02:33 ERROR Connection timeout: ...
2,1,7,2024-01-15 08:03:15 ERROR Connection timeout: ...
3,1,9,2024-01-15 08:05:22 WARNING Disk usage exceede...
4,2,3,2024-01-15 09:05:44 ERROR File not found: /dat...
5,2,4,2024-01-15 09:05:45 ERROR Batch job aborted du...
6,2,5,2024-01-15 09:06:00 WARNING Email notification...
7,2,9,2024-01-15 09:20:00 WARNING SSL certificate ex...


### Breakdown by keyword / キーワード別の内訳

In [5]:
df_multi['level'] = df_multi['text'].apply(
    lambda t: 'ERROR' if 'error' in t.lower() else 'WARNING'
)
print(df_multi.groupby('level').size().rename('count').to_string())

level
ERROR      4
WARNING    4


---
## 4. Export to Excel / Excel出力

Save the results to `data/result.xlsx`. The file can be opened directly in Microsoft Excel or LibreOffice Calc.

In [6]:
# Remove helper column before saving
records_to_save = df_multi[['page', 'line_num', 'text']].to_dict('records')
save_to_excel(records_to_save, 'data/result.xlsx')
print('Done. Results saved to data/result.xlsx')

INFO: Excel出力完了: data/result.xlsx


Done. Results saved to data/result.xlsx
